In [1]:
# 安装需要依赖的库
%pip install trl
%pip install peft
%pip install bitsandbytes

Looking in indexes: http://mirrors.aliyun.com/pypi/simple
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: http://mirrors.aliyun.com/pypi/simple
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: http://mirrors.aliyun.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 2.8 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
# 设置环境变量
%env HF_ENDPOINT=https://hf-mirror.com
%env HF_HOME=/root/autodl-tmp/hf
%env TENSORBOARD_LOGGING_DIR=/root/tf-logs

env: HF_ENDPOINT=https://hf-mirror.com
env: HF_HOME=/root/autodl-tmp/hf
env: TENSORBOARD_LOGGING_DIR=/root/tf-logs


In [3]:
# 加载model和tkennizer
from transformers import AutoModelForCausalLM,AutoTokenizer,BitsAndBytesConfig
import torch
from peft import prepare_model_for_kbit_training

# 4bit量化配置
config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# 模型名称
model_name = "Qwen/Qwen3-8B"

# 加载模型时加载量化配置
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=config)
model = prepare_model_for_kbit_training(model)
tokenizer = AutoTokenizer.from_pretrained(model_name)


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

In [4]:
# 数据集
from datasets import load_dataset

dataset_dict = load_dataset('json', data_files = {"train":"datasets/keywords_data_train.jsonl", "test":"datasets/keywords_data_test.jsonl"})

def map_func(example):
    conversation = example['conversation']
    messages = []
    for item in conversation:
        messages.append({'role':'user','content':item['human']})
        messages.append({'role':'assistant','content':item['assistant']})
    return {'messages':messages}

dataset_dict = dataset_dict.map(map_func,batched=False,remove_columns=['dataset','conversation','category','conversation_id'])

In [5]:
# 微调参数
from trl import SFTConfig,SFTTrainer

from peft import LoraConfig

# Configure LoRA parameters
# r: rank dimension for LoRA update matrices (smaller = more compression)
rank_dimension = 6
# lora_alpha: scaling factor for LoRA layers (higher = stronger adaptation)
lora_alpha = 12
# lora_dropout: dropout probability for LoRA layers (helps prevent overfitting)
lora_dropout = 0.05

peft_config = LoraConfig(
    r=rank_dimension,  # Rank dimension - typically between 4-32
    lora_alpha=lora_alpha,  # LoRA scaling factor - typically 2x rank
    lora_dropout=lora_dropout,  # Dropout probability for LoRA layers
    bias="none",  # Bias type for LoRA. the corresponding biases will be updated during training.
    target_modules="all-linear",  # Which modules to apply LoRA to
    task_type="CAUSAL_LM",  # Task type for model architecture
)

# Configure trainer
training_args = SFTConfig(
    output_dir="/root/autodl-tmp/sft/Qwen3-8B/QLoRA",
    max_steps=1000,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    bf16=True,
    warmup_steps=50,
    logging_dir="/root/tf-logs",
    report_to="tensorboard", 
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["test"],
    processing_class=tokenizer,
    peft_config=peft_config,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [6]:
# 开始训练

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
100,2.232115,2.112811
200,1.908058,2.009704
300,2.164595,1.995936
400,2.032314,1.988933
500,2.092290,1.984103
600,1.930280,1.981130
700,2.112350,1.979275
800,2.021498,1.977553
900,2.031754,1.977056
1000,1.975056,1.976769


TrainOutput(global_step=1000, training_loss=2.1221557750701905, metrics={'train_runtime': 823.828, 'train_samples_per_second': 4.855, 'train_steps_per_second': 1.214, 'total_flos': 4.8189097540608e+16, 'train_loss': 2.1221557750701905})

In [7]:
# 保存最优模型
trainer.save_model('/root/autodl-tmp/sft/Qwen3-8B/QLoRA/best')